# Customer Intelligence & Churn Prediction Platform

## Phase 3: Data Cleaning & Preprocessing

**Objective**

Clean the raw datasets by fixing data types, handling missing values, removing duplicates, and preparing the data for analysis.

**Dataset**

Olist Brazilian E-Commerce Public Dataset

**Author**

Tanish Mhatre

---

## Notebook Roadmap

**1. Load Datasets**
- Load the required datasets.

**2. Inspect Data Quality**
- Check data types, missing values, and duplicates.

**3. Clean Data**
- Fix data types and handle missing values.

**4. Validate Clean Data**
- Verify the cleaned datasets.

**5. Save Processed Data**
- Save cleaned datasets for the next phase.

**6. Summary**
- Document the cleaning steps performed.

In [1]:
# Import pandas
import pandas as pd

# Load datasets
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

# Store datasets
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers
}

**2. Inspect Data Quality**

Check data types, missing values, duplicates, and basic information before cleaning the datasets.

In [7]:
# Check data quality of each dataset

for name, df in datasets.items():

    print("=" * 70)
    print(name.upper())
    print("=" * 70)

    print(f"Shape: {df.shape}")

    print("\nData Types")
    print(df.dtypes)

    print("\nMissing Values")
    print(df.isnull().sum())

    print("\nDuplicate Rows")
    print(df.duplicated().sum())

CUSTOMERS
Shape: (99441, 5)

Data Types
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing Values
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows
0
ORDERS
Shape: (99441, 8)

Data Types
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing Values
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_da

**3. Convert Data Types**

Convert all date columns from string to datetime format for time-based analysis.

In [8]:
# Date columns in orders table

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [11]:
# Convert order date columns to datetime

for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column])


In [12]:
# Verify updated data types

print(orders[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


**Convert Review Date Columns**

Convert review date columns from string to datetime format.

In [15]:
# Review date columns

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

# Convert review date columns to datetime

for column in review_date_columns:
    reviews[column] = pd.to_datetime(reviews[column])

# Verify updated data types

print(reviews[review_date_columns].dtypes)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


**4. Handle Missing Values**

Analyze missing values and decide whether to keep, fill, or remove them based on their business meaning.

In [16]:
# Check missing values

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:
        print("=" * 70)
        print(name.upper())
        print("=" * 70)
        print(missing)

ORDERS
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64
PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


### Missing Value Observations

- Missing order dates are expected for cancelled or undelivered orders.
- Review title and message are optional, so missing values are acceptable.
- Product information contains missing values that require further investigation.

**Investigate Missing Product Records**

View the product records that contain missing values before deciding how to handle them.

In [20]:
# View products with missing values

products[products.isnull().any(axis=1)]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


### Product Missing Value Observation

- 611 products have missing descriptive information.
- Product dimensions are available for most of these records.
- The records will be kept to avoid breaking relationships with other tables.
- Missing values will be handled later if required for analysis or modeling.

**Check Missing Products in Orders**

Verify whether products with missing information are used in customer orders.

In [21]:
# Products with missing values

missing_products = products[
    products.isnull().any(axis=1)
]

# Check if they appear in order_items

used_products = missing_products[
    missing_products["product_id"].isin(order_items["product_id"])
]

print("Missing products:", len(missing_products))
print("Used in orders:", len(used_products))

Missing products: 611
Used in orders: 611


### Product Validation

- All 611 products with missing information are present in customer orders.
- These records will be kept to maintain data integrity.
- Missing values will be handled later if required for analysis.

**Check Duplicate Rows**

Verify whether any dataset contains duplicate records.

In [23]:
# Check duplicate rows

for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"{name}: {duplicates}")

customers: 0
orders: 0
order_items: 0
payments: 0
reviews: 0
products: 0
sellers: 0


### Duplicate Check

- No duplicate rows were found in any dataset.
- No duplicate records were removed.

**Save Cleaned Datasets**

Save the cleaned datasets for use in the next project phase.

In [24]:
customers.to_csv("../data/processed/customers_clean.csv", index=False)
orders.to_csv("../data/processed/orders_clean.csv", index=False)
order_items.to_csv("../data/processed/order_items_clean.csv", index=False)
payments.to_csv("../data/processed/payments_clean.csv", index=False)
reviews.to_csv("../data/processed/reviews_clean.csv", index=False)
products.to_csv("../data/processed/products_clean.csv", index=False)
sellers.to_csv("../data/processed/sellers_clean.csv", index=False)

print("✅ Cleaned datasets saved successfully.")

✅ Cleaned datasets saved successfully.
